<a href="https://colab.research.google.com/github/Umesh-369/Spam-Detection/blob/main/Spam_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Spam Email/SMS Classifier using Support Vector Machine (SVM)
=============================================================
Dataset expected columns: 'Category' (ham/spam) and 'Masseges' (the text)

Steps:
1. Load & clean the dataset
2. Convert text to numerical features using TF-IDF
3. Train an SVM classifier
4. Evaluate performance (accuracy, precision, recall, F1, confusion matrix)
5. Let the user type in a message and get a Spam / Not Spam prediction
"""

import re
import string
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

# -------------------------------------------------------------------
# 1. LOAD DATA
# -------------------------------------------------------------------
DATA_PATH = "/content/spam mail.csv"   # change path if needed

df = pd.read_csv(DATA_PATH)

# Keep only the columns we need and rename them for clarity
df = df[["Category", "Masseges"]].rename(
    columns={"Category": "label", "Masseges": "text"}
)

# Drop any empty rows
df.dropna(subset=["label", "text"], inplace=True)

# Convert labels: ham -> 0 (Not Spam), spam -> 1 (Spam)
df["label_num"] = df["label"].str.strip().str.lower().map({"ham": 0, "spam": 1})


# -------------------------------------------------------------------
# 2. TEXT CLEANING
# -------------------------------------------------------------------
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)          # remove URLs
    text = re.sub(r"\d+", " ", text)                      # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))  # punctuation
    text = re.sub(r"\s+", " ", text).strip()               # extra whitespace
    return text


df["clean_text"] = df["text"].apply(clean_text)


# -------------------------------------------------------------------
# 3. TRAIN / TEST SPLIT
# -------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["label_num"],
    test_size=0.2,
    random_state=42,
    stratify=df["label_num"],
)

# -------------------------------------------------------------------
# 4. FEATURE EXTRACTION (TF-IDF)
# -------------------------------------------------------------------
vectorizer = TfidfVectorizer(stop_words="english", max_df=0.9, min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# -------------------------------------------------------------------
# 5. TRAIN THE SVM MODEL
# -------------------------------------------------------------------
# 'class_weight=balanced' helps because spam (747) is fewer than ham (4825)
svm_model = SVC(kernel="linear", C=1.0, class_weight="balanced", probability=True)
svm_model.fit(X_train_tfidf, y_train)

# -------------------------------------------------------------------
# 6. EVALUATE THE MODEL
# -------------------------------------------------------------------
y_pred = svm_model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
print("=" * 55)
print(f"Model Accuracy : {accuracy * 100:.2f}%")
print("=" * 55)
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=["Not Spam (ham)", "Spam"]))

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# Optional: visualize confusion matrix (comment out if running headless)
try:
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Spam", "Spam"])
    disp.plot(cmap="Blues")
    plt.title("Confusion Matrix - SVM Spam Classifier")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png")
    print("\nConfusion matrix plot saved as 'confusion_matrix.png'")
except Exception as e:
    print("Could not generate plot:", e)


# -------------------------------------------------------------------
# 7. PREDICTION FUNCTION
# -------------------------------------------------------------------
def predict_message(message: str) -> str:
    """Takes a raw email/SMS text and returns 'Spam' or 'Not Spam'."""
    cleaned = clean_text(message)
    vec = vectorizer.transform([cleaned])
    prediction = svm_model.predict(vec)[0]
    probability = svm_model.predict_proba(vec)[0][prediction]
    label = "SPAM" if prediction == 1 else "NOT SPAM"
    return f"{label}  (confidence: {probability * 100:.2f}%)"


# -------------------------------------------------------------------
# 8. INTERACTIVE USER INPUT
# -------------------------------------------------------------------
if __name__ == "__main__":
    print("\n" + "=" * 55)
    print(" SPAM DETECTOR - Enter a message to classify it")
    print(" (type 'exit' or 'quit' to stop)")
    print("=" * 55)

    while True:
        user_input = input("\nEnter an email/message text: ").strip()
        if user_input.lower() in ("exit", "quit"):
            print("Exiting. Goodbye!")
            break
        if not user_input:
            print("Please enter some text.")
            continue

        result = predict_message(user_input)
        print(f"Prediction: {result}")

Model Accuracy : 98.21%

Classification Report:

                precision    recall  f1-score   support

Not Spam (ham)       0.99      0.99      0.99       966
          Spam       0.94      0.92      0.93       149

      accuracy                           0.98      1115
     macro avg       0.97      0.96      0.96      1115
  weighted avg       0.98      0.98      0.98      1115

Confusion Matrix:
 [[958   8]
 [ 12 137]]

Confusion matrix plot saved as 'confusion_matrix.png'

 SPAM DETECTOR - Enter a message to classify it
 (type 'exit' or 'quit' to stop)

Enter an email/message text: Hello
Prediction: NOT SPAM  (confidence: 89.80%)
